# Compliance and PII Scanning

This notebook demonstrates Zipminator's compliance tooling: PII detection,
GDPR-aligned anonymization, and audit trail generation.

These features help organizations meet requirements under **GDPR**, **HIPAA**, and **CCPA**.

- **PIIScanner** detects personally identifiable information across 15+ country patterns
- **AnonymizationEngine** applies 10 levels of anonymization (L1 hashing through L10 full redaction)
- **Audit trails** provide tamper-evident logs for regulatory compliance (DORA Art. 7)

In [ ]:
# PII Scanning on sample patient data
import pandas as pd
from zipminator.crypto.pii_scanner import PIIScanner

scanner = PIIScanner()

# Sample dataset with mixed PII
patient_data = pd.DataFrame({
    "patient_name": ["Alice Johnson", "Bob Smith", "Carol Williams"],
    "email": ["alice@hospital.org", "bob.smith@gmail.com", "carol@clinic.no"],
    "ssn": ["123-45-6789", "987-65-4321", "456-78-9012"],
    "diagnosis": ["Hypertension", "Type 2 Diabetes", "Asthma"],
    "phone": ["555-123-4567", "555-987-6543", "555-246-8135"],
    "dob": ["1990-03-15", "1972-08-22", "1998-11-04"],
})

print("Input Data:")
print(patient_data.to_string(index=False))
print()

# Run PII scan
scan_results = scanner.scan_dataframe(patient_data)

print(f"PII Detected: {scan_results['pii_detected']}")
print(f"Risk Level: {scan_results['risk_level']}")
print(f"Recommended Anonymization Level: {scan_results['recommended_anonymization_level']}")
print(f"Total PII Columns: {len(scan_results['columns_with_pii'])}")
print()
print("Columns with PII:")
for col, matches in scan_results["columns_with_pii"].items():
    print(f"  {col}: {len(matches)} match(es)")
print()
print("Summary:")
print(scan_results["summary"])

In [ ]:
# GDPR-aligned anonymization
from zipminator.crypto.anonymization import AnonymizationEngine

engine = AnonymizationEngine()

# Identify PII columns from scan results
pii_columns = list(scan_results["columns_with_pii"].keys())
print(f"PII columns to anonymize: {pii_columns}")
print()

# GDPR Art. 89 research exemption: L8 for differential-privacy-style anonymization
research_data = engine.apply_anonymization(patient_data.copy(), columns=pii_columns, level=8)
print("GDPR Art. 89 - Research Exemption (L8 Anonymization):")
print(research_data.to_string(index=False))
print()

# HIPAA Safe Harbor / data suppression: L5 for stronger suppression
safe_harbor = engine.apply_anonymization(patient_data.copy(), columns=pii_columns, level=5)
print("HIPAA Safe Harbor (L5 Data Suppression):")
print(safe_harbor.to_string(index=False))

In [ ]:
# Audit trail generation
import json
from datetime import datetime, timezone
from hashlib import sha3_256

audit_entries = [
    {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "operation": "pii_scan",
        "input_hash": sha3_256(patient_data.to_csv().encode()).hexdigest()[:16],
        "pii_detected": scan_results["pii_detected"],
        "risk_level": str(scan_results["risk_level"]),
        "pii_columns_count": len(scan_results["columns_with_pii"]),
        "user": "analyst@corp.com",
    },
    {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "operation": "anonymize",
        "level": 8,
        "purpose": "GDPR Art. 89 research exemption",
        "input_hash": sha3_256(patient_data.to_csv().encode()).hexdigest()[:16],
        "output_hash": sha3_256(research_data.to_csv().encode()).hexdigest()[:16],
        "columns_anonymized": pii_columns,
        "user": "analyst@corp.com",
    },
    {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "operation": "anonymize",
        "level": 5,
        "purpose": "HIPAA Safe Harbor suppression",
        "input_hash": sha3_256(patient_data.to_csv().encode()).hexdigest()[:16],
        "output_hash": sha3_256(safe_harbor.to_csv().encode()).hexdigest()[:16],
        "columns_anonymized": pii_columns,
        "user": "analyst@corp.com",
    },
]

print("Audit Trail")
print("=" * 70)
for entry in audit_entries:
    print(json.dumps(entry, indent=2))
    print()